In [ ]:
import json
import ast
import pandas as pd

In [22]:
import requests
# Function to download files from NOMAD upload
def download_archive(upload_id: str = '8UcL8cCSSYa6sgmhQVXhaQ', file_name='cost_perla_09_06_2026.json'):
    res = requests.get(
        f'https://nomad-lab.eu/prod/v1/api/v1/uploads/{upload_id}/raw/{file_name}?offset=0&length=-1&decompress=false&ignore_mime_type=false&compress=false',
        headers={
            'Accept': 'application/octet-stream',
        },
        timeout=120,
    )
    res.raise_for_status()
    with open(f'{file_name}','wb') as fp:
        fp.write(res.content)
    print("File saved successfully!")

In [24]:
download_archive(file_name='cost_perla_09_06_2026.json')

File saved successfully!


In [15]:
costs_json=json.load(open('cost_perla_09_06_2026.json','r'))
dois=list(costs_json)

In [17]:
costs={i:{'raw_gen_ai_request':[],'litellm_request':[],'Perla Extract - LLM Call Cost':[]} for i in dois}
for doi in dois:
    for trace in costs_json[doi]:
        cost=None
        if trace['type']=='SPAN' and trace['name']=='raw_gen_ai_request':
            cost=ast.literal_eval(trace['metadata']['attributes.llm.anthropic.usage'])
            cost={'input':cost['input_tokens'],'output':cost['output_tokens'],'total':cost['output_tokens']+cost['input_tokens']}
        if trace['type']=='GENERATION' and trace['name']=='litellm_request':
            cost={'usage':trace.get('usageDetails'),'cost':trace.get('costDetails')}
            cost['usage']={k:int(v) for k,v in cost['usage'].items()}
        else:
            try:
                a=json.loads(trace['output'])
                cost={'usage':a.get('usage'),'cost':a.get('costDetails')}
            except:
                pass
        costs[doi][trace['name']].append({'id':trace.get('id'),
                           'parentId':trace.get('parentObservationId'),
                           'type':trace['name'],
                           'cost':cost,
                          'startTime':trace.get('startTime')})
    for i in costs[doi]:
        costs[doi][i]=sorted(costs[doi][i],key=lambda x: x['startTime'])
    if costs[doi]['Perla Extract - LLM Call Cost'][0]['cost'] is None:
        costs[doi]['Perla Extract - LLM Call Cost'][0]['cost']=costs[doi]['litellm_request'][-1]['cost']

In [ ]:
df=[]
for doi, c in costs.items():
    m={'retries':len(c['raw_gen_ai_request'])-1,
    'total_usage':c['Perla Extract - LLM Call Cost'][0]['cost']['usage'],
    'total_cost':c['Perla Extract - LLM Call Cost'][0]['cost']['cost'],
    'usage':[i['cost'] for i in c['raw_gen_ai_request']]}
    dfm={'doi':doi,'retries':len(c['raw_gen_ai_request'])-1}
    dfm.update({f'{k}_tokens':v for k,v in m['total_usage'].items()})
    dfm.update({f'{k}_cost':v for k,v in m['total_cost'].items()})
    for i,c_i in enumerate(m['usage']):
        dfm.update({f'call_{i}_{k}_tokens':v for k,v in c_i.items()})
    df.append(dfm)
df=pd.DataFrame(df).fillna(0)

In [19]:
df

,doi,retries,input_tokens,output_tokens,total_tokens,input_cost,output_cost,total_cost,call_0_input_tokens,call_0_output_tokens,call_0_total_tokens,call_1_input_tokens,call_1_output_tokens,call_1_total_tokens
0,10.1038/s41467-023-36141-8,1,76227,4821,81048,0.228681,0.072315,0.300996,34916,2411,37327,41311.0,2410.0,43721.0
1,10.1002/adfm.202212698,0,32406,5335,37741,0.097218,0.080025,0.177243,32406,5335,37741,0.0,0.0,0.0
2,10.1016/j.jmst.2021.03.045,0,27648,4782,32430,0.082944,0.071730,0.154674,27648,4782,32430,0.0,0.0,0.0
3,10.1021/acsaelm.4c02297,0,31471,2407,33878,0.094413,0.036105,0.130518,31471,2407,33878,0.0,0.0,0.0
4,10.1016/j.matlet.2016.07.004,0,18570,6099,24669,0.055710,0.091485,0.147195,18570,6099,24669,0.0,0.0,0.0
5,10.1038/s41560-022-01102-w,0,32782,5131,37913,0.098346,0.076965,0.175311,32782,5131,37913,0.0,0.0,0.0
6,10.1002/solr.202100879,1,57623,2518,60141,0.172869,0.037770,0.210639,27476,1483,28959,30147.0,1035.0,31182.0
7,10.1038/s41560-022-01061-2,0,15780,1303,17083,0.047340,0.019545,0.066885,15780,1303,17083,0.0,0.0,0.0
8,10.1002/adfm.201904856,0,27601,6971,34572,0.082803,0.104565,0.187368,27601,6971,34572,0.0,0.0,0.0
9,10.1021/acsaem.9b01928,0,29637,2048,31685,0.088911,0.030720,0.119631,29637,2048,31685,0.0,0.0,0.0


In [20]:
agg_df=df[['retries','input_tokens', 'output_tokens', 'total_tokens',
       'input_cost', 'output_cost', 'total_cost']].aggregate(['sum','mean','std']).round(2)

In [21]:
agg_df

,retries,input_tokens,output_tokens,total_tokens,input_cost,output_cost,total_cost
sum,5.00,1073045.00,150633.00,1223678.00,3.22,2.26,5.48
mean,0.17,35768.17,5021.10,40789.27,0.11,0.08,0.18
std,0.38,15611.48,3868.28,17605.63,0.05,0.06,0.09
